In [ ]:
import numpy as np
import pandas as pd
import napari
import imageio.v2 as imageio
from magicgui import magicgui
from qtpy.QtWidgets import QWidget, QVBoxLayout, QLabel, QTextEdit

ANNOTATION_COL = "Phenotype"                  # change to the celltype col
NEW_ANNOTATION_COL = "NewAnnotation"
PROBABILITY_COL = "probability"
REANNOTATED_COL = "Reannotated"
OUTPUT_CSV_PATH = "1_annot_reannotated.csv"

class LabelThresholdWidget(QWidget):
    def __init__(
        self,
        viewer: napari.Viewer,
        seg_layer: napari.layers.Labels,
        annotation_col: str = ANNOTATION_COL,
        new_annotation_col: str = NEW_ANNOTATION_COL,
        probability_col: str = PROBABILITY_COL,
        reannotated_col: str = REANNOTATED_COL,
        output_csv_path: str = OUTPUT_CSV_PATH,
    ):
        super().__init__()
        self.viewer = viewer
        self.seg_layer = seg_layer

        self.annotation_col = annotation_col
        self.new_annotation_col = new_annotation_col
        self.probability_col = probability_col
        self.reannotated_col = reannotated_col
        self.output_csv_path = output_csv_path

        self.current_threshold = 0.2
        self.current_annotation = ""
        self.last_selected_index = None

        self.info_label = QLabel("Selected Label Info:")
        self.info_box = QTextEdit()
        self.info_box.setReadOnly(True)

        @magicgui(
            threshold={"widget_type": "FloatSlider", "min": 0.0, "max": 1.0, "step": 0.01},
            auto_call=True,
        )
        def threshold_control(threshold: float = 0.2):
            self.current_threshold = float(threshold)
            self.apply_filters()

        @magicgui(
            annotation={"widget_type": "ComboBox", "choices": self.annotation_choices()},
            auto_call=True,
        )
        def annotation_control(annotation: str = ""):
            self.current_annotation = annotation
            self.apply_filters()

        @magicgui(
            call_button="Re-annotate Selected Label",
            new_annotation={"widget_type": "ComboBox", "choices": self.annotation_choices()},
        )
        def reannotate_label(new_annotation: str = ""):
            features = self.get_features()
            if features is None:
                print("No features dataframe found.")
                return
            if new_annotation in ("", None):
                print("Please select a valid annotation.")
                return
            if self.last_selected_index is None:
                print("No label selected. Double-click a label first.")
                return

            idx = self.last_selected_index

            if self.new_annotation_col not in features.columns:
                features[self.new_annotation_col] = pd.NA
            if self.reannotated_col not in features.columns:
                features[self.reannotated_col] = False

            old_val = self.effective_annotation(features.loc[idx])
            features.at[idx, self.new_annotation_col] = new_annotation
            features.at[idx, self.reannotated_col] = True

            self.seg_layer.features = features
            self.seg_layer.refresh()
            self.apply_filters()
            self._update_info_display(idx)

            self.annotation_control["annotation"].choices = self.annotation_choices()
            self.reannotate_label["new_annotation"].choices = self.annotation_choices()

            self.save_features_csv()
            print(f"Label index {idx} re-annotated from '{old_val}' to '{new_annotation}'.")

        self.threshold_control = threshold_control
        self.annotation_control = annotation_control
        self.reannotate_label = reannotate_label

        layout = QVBoxLayout()
        layout.addWidget(QLabel("<b>CellDecipher Annotation</b>"))
        layout.addWidget(QLabel("Probability Threshold:"))
        layout.addWidget(self.threshold_control.native)
        layout.addWidget(QLabel("Filter by Annotation:"))
        layout.addWidget(self.annotation_control.native)
        layout.addWidget(QLabel("<b>Re-annotate</b>"))
        layout.addWidget(self.reannotate_label.native)
        layout.addWidget(self.info_label)
        layout.addWidget(self.info_box)
        self.setLayout(layout)

        @self.seg_layer.mouse_double_click_callbacks.append
        def show_label_info(layer, event):
            value = layer.get_value(event.position, world=True)
            if value is None or value == 0:
                self.info_box.setText("No label selected.")
                self.last_selected_index = None
                return

            idx = self.resolve_row_from_clicked_label(value)
            if idx is None:
                self.info_box.setText(f"Clicked label {value}: no matching row in features index.")
                self.last_selected_index = None
                return

            self.last_selected_index = idx
            self._update_info_display(idx)

        self.apply_filters()
        self.viewer.layers.selection.active = self.seg_layer

    def get_features(self):
        return getattr(self.seg_layer, "features", None)

    def resolve_row_from_clicked_label(self, clicked_label):
        """Map clicked segmentation label -> features row index (index is the ID source of truth)."""
        features = self.get_features()
        if features is None:
            return None

        try:
            key = int(clicked_label)
        except Exception:
            key = clicked_label

        if key in features.index:
            return key

        # fallback for string/numeric-mixed indexes
        idx_as_num = pd.to_numeric(pd.Index(features.index), errors="coerce")
        m = idx_as_num == pd.to_numeric(pd.Series([clicked_label]), errors="coerce").iloc[0]
        if m.any():
            return features.index[np.where(m)[0][0]]

        return None

    def effective_annotation(self, row_or_df):
        """Use NewAnnotation when present, else Annotation."""
        if isinstance(row_or_df, pd.Series):
            newv = row_or_df.get(self.new_annotation_col, pd.NA)
            if pd.notna(newv) and str(newv) != "":
                return str(newv)
            return str(row_or_df.get(self.annotation_col, ""))
        else:
            newv = row_or_df.get(self.new_annotation_col, pd.Series(index=row_or_df.index, dtype=object))
            base = row_or_df[self.annotation_col].astype(str)
            return np.where(pd.notna(newv) & (newv.astype(str) != ""), newv.astype(str), base)

    def annotation_choices(self):
        features = self.get_features()
        if features is None or self.annotation_col not in features.columns:
            return [""]
        eff = self.effective_annotation(features)
        vals = pd.Series(eff).dropna().astype(str).unique().tolist()
        return [""] + sorted(vals)

    def _update_info_display(self, idx):
        features = self.get_features()
        if features is None or idx not in features.index:
            self.info_box.setText("No feature info available.")
            return

        row = features.loc[idx].to_dict()
        row["EffectiveAnnotation"] = self.effective_annotation(features.loc[idx])

        text = f"<b>Selected label/index: {idx}</b>\n" + "-" * 40 + "\n"
        for k, v in row.items():
            text += f"{k}: {v}\n"

        is_reannotated = bool(row.get(self.reannotated_col, False))
        text += "-" * 40 + "\n"
        text += (
            "<b style='color: orange;'>⚠ Status: REANNOTATED</b>"
            if is_reannotated
            else "<b style='color: green;'>✓ Status: ORIGINAL</b>"
        )
        self.info_box.setText(text)

    def ids_from_features(self, df: pd.DataFrame) -> np.ndarray:
        """IDs used for masking are always from DataFrame index."""
        if df is None or len(df) == 0:
            return np.array([], dtype=np.int64)
        return pd.to_numeric(pd.Index(df.index), errors="coerce").dropna().astype(np.int64).to_numpy()

    def apply_filters(self):
        features = self.get_features()
        if features is None:
            return
        if self.probability_col not in features.columns or self.annotation_col not in features.columns:
            return

        eff = pd.Series(self.effective_annotation(features), index=features.index)
        f_thresh = features[features[self.probability_col] <= self.current_threshold]

        if self.current_annotation not in ("", None):
            f_ann = features[eff == str(self.current_annotation)]
            keep_ids = np.intersect1d(self.ids_from_features(f_thresh), self.ids_from_features(f_ann))
        else:
            keep_ids = self.ids_from_features(f_thresh)

        seg = self.seg_layer.data
        out = np.zeros_like(seg)
        if keep_ids.size:
            m = np.isin(seg, keep_ids)
            out[m] = seg[m]

        if "Filtered" in self.viewer.layers:
            self.viewer.layers["Filtered"].data = out
        else:
            self.viewer.add_labels(out, name="Filtered")
        self.viewer.layers["Filtered"].contour = 1

    def save_features_csv(self):
        features = self.get_features()
        if features is None:
            return
        out_df = features.copy()
        out_df.to_csv(self.output_csv_path, index=True, index_label="label_index")
        print(f"Saved: {self.output_csv_path}")


if __name__ == "__main__":
    mask_path = "data/myeloma/TS-373_IMC01_UB_001.tiff"     # Change to mask path
    annot_path = "data/myeloma/TS-373_IMC01_UB_001.csv"     # Change to quantification csv path        

    mask = imageio.imread(mask_path).astype(np.uint32)
    labels = pd.read_csv(annot_path)

    # Set index from identifier column (preferred), then use index as the only ID source.
    # Keep the identifier column in the dataframe as a visible feature (drop=False).
    # id_col = next((c for c in ID_COL_CANDIDATES if c in labels.columns), None)
    # if id_col is not None:
    #     labels[id_col] = pd.to_numeric(labels[id_col], errors="coerce")
    #     labels = labels.dropna(subset=[id_col]).copy()
    #     labels[id_col] = labels[id_col].astype(np.int64)
    #     # Keep the ID column as a feature while also using it as index
    #     labels = labels.set_index(id_col, drop=False)
    # else:
    #     labels.index = pd.to_numeric(pd.Index(labels.index), errors="coerce")
    #     labels = labels[~pd.isna(labels.index)].copy()
    #     labels.index = labels.index.astype(np.int64)

    if PROBABILITY_COL not in labels.columns:
        labels[PROBABILITY_COL] = np.round(np.random.rand(len(labels)), 2)
    if ANNOTATION_COL not in labels.columns:
        labels[ANNOTATION_COL] = "Unknown"
    if NEW_ANNOTATION_COL not in labels.columns:
        labels[NEW_ANNOTATION_COL] = pd.NA
    if REANNOTATED_COL not in labels.columns:
        labels[REANNOTATED_COL] = False

    viewer = napari.Viewer()
    seg_layer = viewer.add_labels(mask, name="Segmentation", features=labels, opacity=0.2)
    seg_layer.contour = 1

    combined_widget = LabelThresholdWidget(
        viewer,
        seg_layer,
        annotation_col=ANNOTATION_COL,
        new_annotation_col=NEW_ANNOTATION_COL,
        probability_col=PROBABILITY_COL,
        reannotated_col=REANNOTATED_COL,
        output_csv_path=OUTPUT_CSV_PATH,
    )
    viewer.window.add_dock_widget(combined_widget, area="right")
    viewer.layers.selection.active = seg_layer
    napari.run()

2026-03-31 08:29:58.780 python[48615:64614430] Warning: Window move completed without beginning
